In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [2]:
from datetime import datetime, timedelta

In [3]:
from workflow.pipeline import ephys, mua

c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\datajoint\plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-02-27 17:40:26,310][INFO]: DataJoint 0.14.4 connected to judewerth@db.datajoint.com:3306


Module stfio is not installed. Make sure .abf files are converted to .pkl in Python 2.


In [4]:
# get necessary information
organoid_id = "O09"
start_time = datetime(2023, 5, 4, 4, 12, 52) 
end_time = datetime(2023, 5, 4, 4, 27, 52)

In [5]:
# get ephys key
ephys_key = (ephys.EphysSession & f"organoid_id = '{organoid_id}'" & f"start_time = '{start_time}'" & f"end_time = '{end_time}'").fetch1("KEY")
ephys.EphysSession & ephys_key

organoid_id e.g. O17,experiment_start_time,insertion_number,start_time,end_time,session_type
O09,2023-05-03 17:33:00,0,2023-05-04 04:12:52,2023-05-04 04:27:52,both


In [6]:
# select burst parameters
burst_param_idx = 1

mua.BurstDetectionParamset()

burst_param_idx Unique identifier for the burst detection parameter set,gaus_len_ms Gaussian kernel length in milliseconds,boxcar_len_ms Boxcar kernel length in milliseconds,detection_threshold Threshold for burst detection in standard deviations,min_distance_ms Minimum distance between bursts in milliseconds
1,100,20,2.0,1000.0


In [7]:
burst_session_key = {
    **ephys_key,
    'burst_param_idx': burst_param_idx,
}
burst_session_key

{'organoid_id': 'O09',
 'experiment_start_time': datetime.datetime(2023, 5, 3, 17, 33),
 'insertion_number': 0,
 'start_time': datetime.datetime(2023, 5, 4, 4, 12, 52),
 'end_time': datetime.datetime(2023, 5, 4, 4, 27, 52),
 'burst_param_idx': 1}

In [8]:
mua.BurstSession.insert1(burst_session_key, skip_duplicates=True)

mua.BurstSession & burst_session_key

organoid_id e.g. O17,experiment_start_time,insertion_number,start_time,end_time,burst_param_idx Unique identifier for the burst detection parameter set
O09,2023-05-03 17:33:00,0,2023-05-04 04:12:52,2023-05-04 04:27:52,1


In [9]:
mua.PopulationBursts.populate(burst_session_key)

{'success_count': 1, 'error_list': []}

In [10]:
mua.PopulationBursts & burst_session_key

organoid_id e.g. O17,experiment_start_time,insertion_number,start_time,end_time,burst_param_idx Unique identifier for the burst detection parameter set,burst_indices ms since start of detected bursts within the time frame,burst_peak_heights Peak heights of detected bursts,"burst_bounds [-ms, +ms] relative to burst peak (firing rate >= 10% of peak height)",burst_spike_array Single electrode spike array for each burst (num_bursts x num_electrodes x time_window),weighted_sttc Spike time tiling coefficient across all electrodes weighted by number of spikes
O09,2023-05-03 17:33:00,0,2023-05-04 04:12:52,2023-05-04 04:27:52,1,=BLOB=,=BLOB=,=BLOB=,=BLOB=,=BLOB=
